# Agent Comparison

This notebook compares different agent types on the same puzzle environment to demonstrate their relative performance.

**Agents compared:**
- **PPOAgent**: Simple table-based PPO (no dependencies)
- **GRPOAgent**: Group Relative Policy Optimization (no dependencies)
- **NearestNeighborAgent**: k-NN baseline (no dependencies)
- **PPOAgentTorch**: PyTorch neural network PPO (requires `torch`)
- **GRPOAgentTorch**: PyTorch neural network GRPO (requires `torch`)
- **RNNAgent**: LSTM-based agent with memory (requires `torch`)
- **TransformerAgent**: Self-attention agent (requires `torch`)

In [ ]:
import sys
sys.path.insert(0, '../..')

from envgen.environments.base import EnvironmentConfig
from envgen.environments.puzzle_env import PuzzleEnvironment
from envgen.evaluators.skill_evaluator import SkillEvaluator
from envgen.utils.types import DifficultyLevel, Episode, Transition
import time

In [ ]:
def train_and_evaluate_agent(agent, num_episodes=15, num_cycles=3):
    """Train an agent and return evaluation results."""
    config = EnvironmentConfig(
        name="comparison_env",
        difficulty=DifficultyLevel.EASY,
        parameters={"grid_size": 6, "num_items": 2, "num_obstacles": 3},
        max_steps=50,
    )
    
    all_episodes = []
    start = time.time()
    
    for cycle in range(num_cycles):
        env = PuzzleEnvironment(config=config)
        episodes = []
        
        for _ in range(num_episodes):
            env.reset()
            transitions = []
            ep_reward = 0.0
            
            while not env.done:
                obs = env.get_observation()
                available = env.get_available_actions()
                action = agent.select_action(obs, available)
                next_obs, reward, done, info = env.step(action)
                
                transitions.append(Transition(
                    observation=obs, action=action, reward=reward,
                    next_observation=next_obs, done=done
                ))
                ep_reward += reward
            
            episodes.append(Episode(transitions=transitions, total_reward=ep_reward))
        
        all_episodes.extend(episodes)
        if episodes:
            agent.train(episodes)
    
    elapsed = time.time() - start
    
    # Evaluate
    evaluator = SkillEvaluator()
    result = evaluator.evaluate(all_episodes, agent_id=agent.name)
    
    avg_reward = sum(e.total_reward for e in all_episodes) / len(all_episodes)
    
    return {
        "agent": agent.name,
        "avg_reward": round(avg_reward, 3),
        "overall_skill": round(result.overall_score, 3),
        "success_rate": round(result.success_rate, 3) if hasattr(result, 'success_rate') else 0.0,
        "time_seconds": round(elapsed, 2),
    }

In [ ]:
# Compare agents that don't require external dependencies
agents_to_compare = []

from envgen.agents import PPOAgent, GRPOAgent, NearestNeighborAgent

agents_to_compare.append(PPOAgent(name="PPOAgent", config={"learning_rate": 0.1, "epsilon": 0.2}))
agents_to_compare.append(GRPOAgent(name="GRPOAgent", config={"learning_rate": 0.1, "epsilon": 0.2, "group_size": 4}))
agents_to_compare.append(NearestNeighborAgent(name="NearestNeighborAgent", config={"k": 5}))

# Try PyTorch agents if available
try:
    from envgen.agents import PPOAgentTorch
    agents_to_compare.append(PPOAgentTorch(name="PPOAgentTorch", config={"learning_rate": 0.001, "hidden_size": 64}))
except ImportError:
    print("PPOAgentTorch not available (install torch)")

try:
    from envgen.agents import GRPOAgentTorch
    agents_to_compare.append(GRPOAgentTorch(name="GRPOAgentTorch", config={"learning_rate": 0.001, "hidden_size": 64}))
except ImportError:
    print("GRPOAgentTorch not available (install torch)")

try:
    from envgen.agents import RNNAgent
    agents_to_compare.append(RNNAgent(name="RNNAgent", config={"learning_rate": 0.001, "hidden_size": 64}))
except ImportError:
    print("RNNAgent not available (install torch)")

try:
    from envgen.agents import TransformerAgent
    agents_to_compare.append(TransformerAgent(name="TransformerAgent", config={"learning_rate": 0.001, "hidden_size": 64}))
except ImportError:
    print("TransformerAgent not available (install torch)")

print(f"Comparing {len(agents_to_compare)} agents...")

In [ ]:
# Run comparison
results = []
for agent in agents_to_compare:
    print(f"Training {agent.name}...")
    result = train_and_evaluate_agent(agent)
    results.append(result)
    print(f"  Avg reward: {result['avg_reward']:.3f}")
    print(f"  Overall skill: {result['overall_skill']:.3f}")
    print(f"  Time: {result['time_seconds']:.2f}s")
    print()

In [ ]:
# Print summary table
print("=" * 80)
print("AGENT COMPARISON SUMMARY")
print("=" * 80)
print(f"{'Agent':<25}  {'Avg Reward':>12}  {'Skill Score':>13}  {'Success Rate':>14}  {'Time (s)':>10}")
print("-" * 80)

for r in results:
    print(f"{r['agent']:<25}  {r['avg_reward']:>12.3f}  {r['overall_skill']:>13.3f}  {r['success_rate']:>14.3f}  {r['time_seconds']:>10.2f}")

print("=" * 80)

# Find best agent by skill score
best = max(results, key=lambda x: x['overall_skill'])
print(f"\nBest agent by skill score: {best['agent']} ({best['overall_skill']:.3f})")